# AT-TPC Latent Space Exploration
---
This notebook provides an interactive framework to analyze the high-dimensional structural clustering patterns of AT-TPC physics events embeddings.

In [3]:
import os
import sys
import numpy as np
import random
sys.path.append('../../')
from clustering import t_SNE_clustering
from clustering import UMAP_embedding
from clustering import k_means_clustering
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

In [4]:
# create a folder for results of global feature exploration

folder_path = './plots/exploration'
if not os.path.exists(folder_path):
  os.makedirs(folder_path, exist_ok=True)

## 1. Data Ingestion
---
In this phase, we ingest the .npy files that hold the embeddings from each team member's model training, and the labels which are then split into 3 different groups. 

| Grouping Range | Target Assignment Value | 
| :--- | :---: |
| **0, 1, or 2 Tracks** | `0` |
| **3 Tracks** | `1` |
| **4 or 5 Tracks** | `2` |



In [5]:
# load the global features
features = np.load('../../data/features.npy') # CHANGE THE NAME OF THE DATA FILE AS NEEDED
label_names = ["0,1,2 Tracks", "3 Tracks", "4, 5 Tracks"]

print(f"Features loaded successfully! Shape: {features.shape}")

Features loaded successfully! Shape: (200, 1024)


In [6]:
# load labels
label_data = np.load('../../data/master_labels.npy') # CHANGE THE NAME OF THE DATA FILE AS NEEDED  


In [7]:
# Clean conversion directly to native flat arrays
labels = label_data.astype(int)

# Regroup physical track categories uniformly
labels = np.where(np.isin(labels, [0, 1, 2]), 0, labels)
labels = np.where(np.isin(labels, [3]), 1, labels)
labels = np.where(np.isin(labels, [4, 5]), 2, labels)

features = StandardScaler().fit_transform(features)

print(f"Data scaled successfully! {features.shape}")

Data scaled successfully! (200, 1024)


## 2. K-Means 
---
We run the K-Means algorithm with k=3 on the full set of embeddings. Grouping our embeddings into k distinct categories.

In [8]:
# Create the internal hardcoded folder structure so the function doesn't crash on save
os.makedirs('./plots/k_means/2d_plots', exist_ok=True)

save_dir = './plots/k_means'
# 1. features, 2. labels, 3. dimension, 4. save_dir, 5. num_samples_to_print=10
features_glob, cluster_labels, cluster_indices = k_means_clustering(
    features, 
    labels, 
    2, 
    save_dir, 
    10
)


Random sample indices from each cluster:
Cluster 0: [95, 8, 46, 130, 156, 23, 171, 196, 51, 21]


## 3. PCA Projection
 ---
 Before running nonlinear methods, we establish a linear PCA baseline.
 PCA captures the directions of maximum variance in the 1024-D feature space
 and projects them down

In [9]:
from clustering import pca_clustering

os.makedirs('./plots/pca/2d_plots', exist_ok=True)
os.makedirs('./plots/pca/3d_plots', exist_ok=True)

save_dir = './plots/pca'

_, _ = pca_clustering(
    features,
    labels,
    2,          # 2D projection
    save_dir
)

ImportError: cannot import name 'pca_clustering' from 'clustering' (/home/DAVIDSON/joarenas/projects/ATTPCLatent/latent_layer_processing/global_feature_exploration/clustering.py)

## 4. Mapping Cluster Assignments back to Original Events
---
After k-means assigns each embedding to a cluster, we record which original event indices 
belong to each cluster. This lets us look up specific events by cluster.

In [ ]:
def transform_ids(cluster_ids, original_ids):
    original = []

    for index in cluster_ids:
        original.append(original_ids[index])
    return original

sequential_row_ids = np.arange(len(features))
clusters = []

print("Transformed indices:")
for i, cluster in enumerate(cluster_indices):
    transformed_cluster = transform_ids(cluster, sequential_row_ids)
    clusters.append(transformed_cluster)
    print(f"Cluster {i}: {transformed_cluster}")

## 5. Check Cluster Accuracy with Real Labels
---
We check how accurate the automatic groups are by printing the actual track labels for the events inside each cluster. This lets us verify if the geometric grouping matches our physics tracking expectations.

In [ ]:
def true_labels(indices, labels):
    c_labels = []
    for index in indices:
        c_labels.append(labels[index])

    return c_labels

print("True labels verification per cluster:")
for i, cluster in enumerate(cluster_indices):
    print(f"Cluster {i}: {true_labels(cluster, labels)}")

## 6. Visualize the Latent Space Using UMAP
---
We run the UMAP algorithm on our entire feature matrix all at once. In this representation, global structure does matter.

The raw 2D coordinates are saved into `./plots/exploration/umap_data/` 

In [ ]:
os.makedirs('./plots/umap/2d_plots', exist_ok=True)

neighbors = 50
plt_colors = ['blue', 'green', 'red']
label_names = ["0,1,2 Tracks", "3 Tracks", "4, 5 Tracks"]

fig, ax = plt.subplots(figsize=(10, 7))

umap_results = UMAP_embedding(
    features,
    2,
    ax,
    labels,
    label_names,
    plt_colors,
    neighbors
)

ax.legend()
plt.title(f"UMAP 2D Manifold Embedding (n_neighbors={neighbors})")
plt.savefig('./plots/umap/umap_unified_manifold.png', dpi=200)
plt.show()

## 7. Visualize The Latent Space Using t-SNE
---
Finally, we run t-SNE on our full dataset all at once. In this representation, global structure does not give us any information.

The raw 2D coordinates are saved into `./plots/exploration/tsne_data/`

In [ ]:
os.makedirs('./plots/t-sne/2d_plots', exist_ok=True)

perplexity = 40
plt_colors = ['blue', 'green', 'red']
label_names = ["0,1,2 Tracks", "3 Tracks", "4, 5 Tracks"]

fig, ax = plt.subplots(figsize=(10, 7))

tsne_results = t_SNE_clustering(
    features,
    2,
    ax,
    labels,
    label_names,
    plt_colors,
    perplexity
)

ax.legend()
plt.title(f"t-SNE 2D Manifold Embedding (perplexity={perplexity})")
plt.savefig('./plots/t-sne/tsne_unified_manifold.png', dpi=200)
plt.show()

## 8. Silhouette Score 
---
To quantitatively validate the visual cluster separation observed in the UMAP and t-SNE projections,
we compute a silhouette score directly on the original 1024-D feature space.

A score near **1.0** indicates tight, well-separated clusters. A score near **0.0** indicates
overlapping or poorly defined structure.

In [ ]:
from sklearn.metrics import silhouette_score

# Compute on original 1024-D features, not 2D projections
# sample_size avoids expensive full computation at high dimensionality
sil_score = silhouette_score(features, label_data, sample_size=2000, random_state=42)

print("=" * 50)
print(f"Silhouette Score (1024-D latent space): {sil_score:.4f}")
print("=" * 50)

if sil_score >= 0.7:
    print("Interpretation: Strong, well-separated cluster structure")
elif sil_score >= 0.5:
    print("Interpretation: Reasonable cluster structure")
elif sil_score >= 0.25:
    print("Interpretation: Weak cluster structure — representations may need improvement")
else:
    print("Interpretation: No meaningful separation — contrastive objective needs tuning")